#Transform Circuits Data
 1. Read Bronze Circuits Data
 2. Keep only columns required for analytics ( Drop url column)
 3. Standardise column names using snake_case(circuitId -> circuit_Id, circuitName -> circuit_name)
 4. Rename Columns to make them more meaningful (lat->latitude, long->longitude)
 5. Filter out rows where circuit_Id is null
 6. Remove duplicate records
 7. Transform values of columns circuit_name and locality to Title case
 8. Write the transformed data to silver circuits table


In [0]:
%run ../00_common/01_environment_config

In [0]:
%run ../00_common/01_environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
circuits_read_df = spark.table(bronze_table)
display(circuits_read_df)

In [0]:
from pyspark.sql import functions as F
circuits_select_df = circuits_read_df.select(
    F.col("circuitId").alias("circuit_id"),
    F.col("circuitName").alias("circuit_name"),
    F.col("lat").alias("latitude"),
    F.col("long").alias("longitude"),
    F.col("locality"),
    F.col("country"),
     F.col("ingestion_timestamp"),
     F.col("source_file")
    )

In [0]:
from pyspark.sql import functions as F
filter_circuits_df = circuits_select_df.filter(F.col("circuit_id").isNotNull())
display(filter_circuits_df)

In [0]:
from pyspark.sql import functions as F
rm_dup_circuits_df = filter_circuits_df.dropDuplicates(["circuit_id"])
display(rm_dup_circuits_df)

In [0]:
from pyspark.sql import functions as F
circuits_final_df = (
    rm_dup_circuits_df
    .withColumn("circuit_id", F.initcap(F.col("circuit_id")))
    .withColumn("locality", F.initcap(F.col("locality")))
)
display(circuits_final_df)

In [0]:
(circuits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)